# Modelo: Predicción de inversión en una tienda de videojuegos

Este cuaderno entrena el modelo de **regresión** y guarda el archivo `modelo-reg.pkl` que usa la app de Streamlit (`app.py`).

Pasos:
1. Cargar los datos de entrenamiento
2. Preparar los datos: limpieza, dummies y normalización
3. Entrenar y evaluar el modelo (MAPE)
4. Guardar el modelo: `[modelo, min_max_scaler, variables]`

## 1. Cargar los datos
Sube tu archivo de entrenamiento a Colab (ícono de carpeta a la izquierda) y escribe su nombre y el de la variable objetivo.

In [ ]:
import numpy as np
import pandas as pd
import pickle

ARCHIVO = "videojuegos.csv"      # <- nombre de tu archivo de entrenamiento
OBJETIVO = "Presupuesto"         # <- columna que quieres predecir (la inversión)

data = pd.read_csv(ARCHIVO)
print(data.shape)
print("Columnas:", list(data.columns))
data.head()

In [ ]:
# Verificación: la columna objetivo debe existir
if OBJETIVO not in data.columns:
    raise ValueError(f"No existe la columna '{OBJETIVO}'. Cambia OBJETIVO por una de estas: {list(data.columns)}")

data.info()

## 2. Preparación de los datos

In [ ]:
# Variables que usa la app de Streamlit (deben llamarse igual)
predictoras = ["Edad", "videojuego", "Plataforma", "Sexo", "Consumidor_habitual"]
categoricas = ["videojuego", "Plataforma", "Sexo", "Consumidor_habitual"]

data = data[predictoras + [OBJETIVO]].copy()

# Limpieza: quitar registros con datos faltantes
print("Faltantes por columna:")
print(data.isnull().sum())
data = data.dropna()

# Consumidor_habitual como texto 'True'/'False', igual que en la app
data["Consumidor_habitual"] = data["Consumidor_habitual"].astype(str)
print("Registros después de limpiar:", len(data))

In [ ]:
# IMPORTANTE: los valores de la app deben ser idénticos a los del dataset
# (incluidas las comillas simples). Revisa que coincidan con las listas de app.py.
for col in categoricas:
    print(col, "->", sorted(data[col].unique()))

In [ ]:
# Dummies: en el ENTRENAMIENTO se usa drop_first=True
# (en el despliegue se usa drop_first=False y luego reindex con 'variables')
X = pd.get_dummies(data[predictoras], columns=categoricas, drop_first=True, dtype=int)
y = data[OBJETIVO]

variables = X.columns          # lista de columnas que guardaremos con el modelo
print(len(variables), "variables:")
print(list(variables))

In [ ]:
# Normalización de la edad (necesaria para Regresión, KNN, Red y SVM)
from sklearn.preprocessing import MinMaxScaler

min_max_scaler = MinMaxScaler()
X[["Edad"]] = min_max_scaler.fit_transform(X[["Edad"]])   # fit SOLO en el entrenamiento
X.head()

## 3. Entrenamiento y evaluación

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, r2_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelo = LinearRegression()
modelo.fit(X_train, y_train)

y_pred = modelo.predict(X_test)
mape = mean_absolute_percentage_error(y_test, y_pred)
print(f"MAPE: {mape:.2%}   (error porcentual promedio)")
print(f"MAE : {mean_absolute_error(y_test, y_pred):,.2f}")
print(f"R²  : {r2_score(y_test, y_pred):.3f}")

In [ ]:
# Comparación visual: valor real vs predicho
import matplotlib.pyplot as plt

plt.figure(figsize=(5, 5))
plt.scatter(y_test, y_pred, alpha=0.6)
lim = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(lim, lim, "r--")
plt.xlabel("Valor real")
plt.ylabel("Valor predicho")
plt.title("Regresión lineal: real vs predicho")
plt.show()

In [ ]:
# Modelo final: se reentrena con TODOS los datos para el despliegue
modelo = LinearRegression()
modelo.fit(X, y)

# Coeficientes (cuánto aporta cada variable a la predicción)
pd.Series(modelo.coef_, index=variables).sort_values()

## 4. Guardar el modelo
Se guardan juntos el modelo, el escalador y la lista de variables, en el orden que lee `app.py`.

Actualiza el mensaje de error en `app.py` con el MAPE que obtuviste.

In [ ]:
filename = "modelo-reg.pkl"
with open(filename, "wb") as f:
    pickle.dump([modelo, min_max_scaler, variables], f)
print("Modelo guardado en", filename)

import sklearn
print("Versión de scikit-learn (escríbela en requirements.txt):", sklearn.__version__)

In [ ]:
# Prueba rápida: cargar el .pkl y predecir un cliente, igual que hará la app
modelo2, scaler2, variables2 = pickle.load(open(filename, "rb"))

nuevo = pd.DataFrame([[25, data["videojuego"].iloc[0], data["Plataforma"].iloc[0], "Mujer", "True"]],
                     columns=predictoras)
nuevo_prep = pd.get_dummies(nuevo, columns=categoricas, drop_first=False, dtype=int)
nuevo_prep = nuevo_prep.reindex(columns=variables2, fill_value=0)
nuevo_prep[["Edad"]] = scaler2.transform(nuevo_prep[["Edad"]])
print("Predicción de prueba:", modelo2.predict(nuevo_prep)[0])

In [ ]:
# Descargar el archivo para subirlo a GitHub junto con app.py
from google.colab import files
files.download(filename)